# 🔧 Lab 3 — Build & Deploy the Maintenance Cockpit App

In **Lab 1** you created the factory data; in **Lab 2** you served it from Lakebase. Now you'll put a **real web app** in front of it — and actually understand how **Databricks Apps** work.

## What *is* a Databricks App? 🤔
A **fully-managed web app** (Streamlit, Dash, Flask, Node, …) that runs **on Databricks compute, right next to your data**, behind workspace **SSO**, with its **own identity**. There are **no servers to provision, no hosting to buy, no secrets to manage** — Databricks runs it, secures it, and gives it *governed* access to your data.

## What you'll do here
- Deploy the **Maintenance Cockpit** (a Streamlit app) and connect it to *your* Lakebase database
- Understand the moving parts: the app's **code**, the difference between **create and deploy**, its **identity & authentication**, how it reaches **Lakebase + the Lakehouse**, and where to **watch it in the UI**
- Finish the one piece left for you — the **write-back** — and see the round-trip come alive

## Why it's cool 💡
The people who act on data in real time — a technician at 2 a.m. — can't work off a BI dashboard. An App serves them the data **operationally** *and* captures what they do. And because it's all governed by Unity Catalog, their actions flow **straight back into the lakehouse**. One platform, no second system to secure, no ETL in between.

⏱️ **~30 min · 100% browser** (no CLI, nothing to install).

## 🗺️ The big picture — where the app sits

The app has four tabs (**Alerts & actions**, **Work orders**, **Quality checks**, **Operator notes**), each reading and writing live to your Lakebase Postgres:

```
 Lakehouse (Unity Catalog)          Lakebase (Postgres)                 This App
 ┌───────────────────────┐  Lab 2  ┌────────────────────────┐          ┌───────────────────┐
 │ machines, tickets …   │ ──sync─▶│ synced tables (read)   │ ──read──▶│ 🔴 shows alerts   │
 │ (governed source)     │         │  in lakebase_<you>     │          │                   │
 │                       │         │ operational tables     │ ◀─write──│ ✍️ log a fix      │
 │ lb_*_history      ◀───┼──CDF────┤  in public             │          └───────────────────┘
 └───────────────────────┘ (Lab 4) └────────────────────────┘
```

- **Lakehouse = the governed source of truth** (Delta tables in Unity Catalog).
- **Lakebase = the operational serving layer** the app actually talks to (millisecond Postgres reads/writes).
- The app **never queries the Lakehouse directly.** It reads the Lakebase replicas Lab 2 created; its writes stream back to the Lakehouse via **Change Data Feed** (that's Lab 4).

## 📁 Where does the app's code come from?

An App is **multi-file Python that you deploy as a folder** — so it lives as files in this repo at **[`bundle/src/app/`](../bundle/src/app)**, and you deploy that whole folder. Four files matter:

| File | What it is |
|------|------------|
| **`app.py`** | The **Streamlit UI** — the four tabs, the buttons, the tables you see in the browser. |
| **`db.py`** | **All data access** — opens the Postgres connection, runs the read queries, and holds the write-back you'll finish in Step 5. |
| **`app.yaml`** | The **run config** — the command that starts it (`streamlit run app.py`), the port, and the Lakebase coordinates. **No password** (see auth, next). |
| **`requirements.txt`** | Python dependencies Databricks installs for you at deploy time. |

> 💡 Open `bundle/src/app/app.py` and `db.py` from the repo folder and skim them now — you'll edit `db.py` in Step 5.

## 🔐 How the app logs in and reaches your data

This is the part that makes Apps click:

**The app has its own identity.** When you *create* the app (Step 2), Databricks gives it a **service principal** — a machine identity. The app runs **as that service principal**, never as you.

**No passwords, ever.** `db.py` mints a **fresh, short-lived OAuth token per connection** via `w.postgres.generate_database_credential(ENDPOINT_NAME)`. That's why `app.yaml` has **no `PGPASSWORD`** — credentials are generated on the fly and expire in ~1 hour.

**Reaching Lakebase** — two things you'll do:
1. **Bind** the Lakebase database to the app as a **`postgres` resource** → Databricks provisions a Postgres role for the app's service principal and injects `PGHOST`, `PGDATABASE`, and the SP's `DATABRICKS_CLIENT_ID` into the running app.
2. **Grant** that service principal read/write on your tables (a one-time SQL grant).

**Reaching the Lakehouse** — *indirectly, and that's the point.* The data is born in **Unity Catalog** (the Lakehouse), Lab 2 **synced** it into Lakebase, and the app reads those Lakebase replicas. Because Lakebase is registered in UC, everything stays **governed**, and the app's writes flow back to the Lakehouse via CDF (Lab 4). The app itself only ever speaks **Postgres**.

> 💡 Your read-only reference tables live in the `lakebase_<you>` schema and the writable operational ones in `public`. `db.py` sets the connection's `search_path` to both, so every query uses plain table names — no schema prefixes.

### Service principal vs. on-behalf-of-user (OBO) — which and when? 🔀
This app connects as its **service principal (SP)**: one shared identity, one set of grants, everyone sees the same operational data. That's the right default for a shared cockpit.

Databricks Apps can also run **on behalf of the signed-in user (OBO)** — you declare authorization **scopes**, and the app acts as *that user*. Then Lakebase/UC access is governed by the **user's own** permissions (they mint the DB credential as themselves).

| Use **SP** when… | Use **OBO** when… |
|---|---|
| Shared data, everyone sees the same thing | Each user may only see *their* data (row/column security) |
| Simplest: grant the SP once | You need per-user authorization / audit-as-the-user |

## Step 1 — Set up & point the app at your Lakebase 🧭

First install the two helpers and restart Python. Then the config cell **finds your own Lakebase project** (the `lakebase-ws-<you>-N` from Lab 2) and writes its coordinates into `app.yaml` for you — nothing to type, it even detects the repo path itself.

> ⚠️ `restartPython()` clears everything, so run the cell right after it.

In [ ]:
%pip install -U "databricks-sdk>=0.118.0" "psycopg[binary]>=3.1" -q

In [ ]:
dbutils.library.restartPython()

In [ ]:
import re, time, pathlib
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.apps import App, AppDeployment
import psycopg

# Find the app automatically — search from this notebook's folder upward for bundle/src/app.
# (Works whether the notebook sits in <repo>/notebooks/ or right next to bundle/. Nothing to type.)
_dir = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get().rsplit("/", 1)[0]
REPO = None
for _ in range(5):
    _cand = _dir if _dir.startswith("/Workspace") else "/Workspace" + _dir
    if pathlib.Path(f"{_cand}/bundle/src/app").exists():
        REPO = _cand
        break
    if _dir.count("/") <= 1:
        break
    _dir = _dir.rsplit("/", 1)[0]
assert REPO, ("Couldn't find bundle/src/app near this notebook. Make sure the repo's bundle/ "
              "folder sits next to this notebook (or one level up).")
APP_SRC = f"{REPO}/bundle/src/app"

w = WorkspaceClient()
user = w.current_user.me().user_name
slug = re.sub(r"[^a-z0-9]", "", user.split("@")[0].lower())

# Find the Lakebase project you created in Lab 2 (named lakebase-ws-<you>-N)
PROJECT = None
for i in range(1, 21):
    cand = f"lakebase-ws-{slug}-{i}"
    try:
        w.postgres.get_project(name=f"projects/{cand}")
        list(w.postgres.list_branches(parent=f"projects/{cand}"))   # is it healthy?
        PROJECT = cand
        break
    except Exception:
        continue
assert PROJECT, "No Lakebase project found — did you finish Lab 2?"

BRANCH   = f"projects/{PROJECT}/branches/production"
ENDPOINT = f"{BRANCH}/endpoints/primary"
PGDB     = "databricks_postgres"          # your operational tables live here (public schema)
LBSCHEMA = f"lakebase_{slug}"             # your read-only synced tables live here
APP      = ("lb-workshop-" + slug)[:30].rstrip("-")
host = next(e for e in w.postgres.list_endpoints(parent=BRANCH)
            if e.name == ENDPOINT).status.hosts.host

# Fill your endpoint + host into the app's config file (no hand-editing needed)
def _set(t, k, v):
    return re.sub(rf'(- name: {k}\n\s*value: )"[^"]*"', rf'\g<1>"{v}"', t)
cfg = pathlib.Path(f"{APP_SRC}/app.yaml")
txt = cfg.read_text()
txt = _set(txt, "ENDPOINT_NAME", ENDPOINT)
txt = _set(txt, "PGHOST", host)
cfg.write_text(txt)

print(f"✅ Repo detected:               {REPO}")
print(f"✅ Found your Lakebase project: {PROJECT}")
print(f"   Your app will be named:     {APP}")
print(f"   Synced tables are in schema: {LBSCHEMA}")
print(f"   App config points at:        {host}")

## Step 2 — Create the app  (create ≠ deploy!) 🏗️

People mix these two up — here's the difference:

| | **Create** (this step) | **Deploy** (Step 4) |
|---|---|---|
| What it does | Provisions the app: its **compute**, its **service principal**, and attaches **resources** (your Lakebase DB) | Uploads a **version of your code** to the app and (re)starts it |
| How often | **Once** | **Every time your code changes** |
| Think of it as | Set up the machine + identity + plug in the database | Ship the code onto it |

So *create* gives you an empty, fully-wired app; *deploy* puts your code on it. The cell below **creates the app and binds your Lakebase database** in one go (first run provisions a little compute, ~2 min).

> 🖱️ **Prefer clicking?** UI path: **Compute ▸ Apps ▸ Create app**, then the app's **Resources** tab ▸ *Add resource ▸ Database* to bind Lakebase. This cell does exactly that, scripted.

In [ ]:
# Which Postgres database object to bind (the one Lab 2 created)
dbres = next(d.as_dict()["name"] for d in w.postgres.list_databases(BRANCH)
             if d.as_dict()["status"]["postgres_database"] == PGDB)

try:
    w.apps.get(name=APP)
    print(f"✅ App '{APP}' already exists — skipping create")
except Exception:
    print(f"⏳ Creating '{APP}' and giving it a little compute — usually ~2 minutes. "
          f"You'll see progress below (the notebook stays responsive).")
    # Kick off creation WITHOUT blocking the kernel, then poll politely.
    w.apps.create(App.from_dict({
        "name": APP,
        "description": "Maintenance Cockpit — Lakebase in a Day",
        "resources": [{
            "name": "lakebase-db",
            "postgres": {"branch": BRANCH, "database": dbres,
                         "permission": "CAN_CONNECT_AND_CREATE"},
        }],
    }))
    for _ in range(90):
        try:
            state = w.apps.get(name=APP).as_dict().get("compute_status", {}).get("state")
        except Exception:
            state = "PROVISIONING"
        if state == "ACTIVE":
            break
        print(f"   compute: {state} …")
        time.sleep(10)
    print("✅ App created and connected to your Lakebase database")

## Step 3 — Grant the app access to your tables 🔑

Binding lets the app *connect*, but not yet *read your data*. Grant its service principal, once:
- **`pg_read_all_data`** — read every table (synced + operational), now and after any re-sync
- **`pg_write_all_data`** — write the operational tables (that's the technician's work)
- **`USAGE, SELECT ON … SEQUENCES`** — so inserts can use the `SERIAL` primary keys

> 🖱️ **In the UI:** these are plain SQL — run them in the **SQL editor**. (There's no button for grants; they're standard Postgres permissions.)

> 🔎 **Verify what the app can do** — the cell prints the app's `service_principal_client_id`; run this in the **Lakebase SQL editor** (replace `<sp>`) to see its privileges:
> ```sql
> -- role memberships (expect pg_read_all_data + pg_write_all_data)
> SELECT r.rolname FROM pg_auth_members m
>   JOIN pg_roles r ON r.oid = m.roleid
>   JOIN pg_roles u ON u.oid = m.member
> WHERE u.rolname = '<sp>';
> -- explicit table grants
> SELECT table_schema, table_name, privilege_type
> FROM information_schema.role_table_grants WHERE grantee = '<sp>';
> ```
> (In `psql`, `\du <sp>` shows the same role memberships.)

In [ ]:
# The app runs as its own identity (a "service principal"). Give that identity read access to
# your synced tables and read+write to your operational ones — once.
#   • pg_read_all_data / pg_write_all_data cover every table, now and after a re-sync
#   • the sequence grant lets inserts use the SERIAL primary keys
# (You don't need to touch search_path here — the app sets its own per connection, so it finds
#  tables in both the public and lakebase_<you> schemas automatically.)
sp = w.apps.get(name=APP).as_dict()["service_principal_client_id"]

with psycopg.connect(host=host, port=5432, dbname=PGDB, user=user,
                     password=w.postgres.generate_database_credential(ENDPOINT).token,
                     sslmode="require", autocommit=True) as c:
    c.execute(f'GRANT USAGE ON SCHEMA public TO "{sp}"')
    c.execute(f'GRANT pg_read_all_data  TO "{sp}"')
    c.execute(f'GRANT pg_write_all_data TO "{sp}"')
    c.execute(f'GRANT USAGE, SELECT ON ALL SEQUENCES IN SCHEMA public TO "{sp}"')

print("✅ Permissions granted — the app can read your synced tables and read/write the operational ones.")

## Step 4 — Deploy & open your app 🚀

Now ship it. The cell pushes the code and starts the app (non-blocking, with progress) and prints the URL.

**✅ Check:** deploy reaches **SUCCEEDED**. Open the URL — you'll see the four tabs and the **open alerts** pulled live from your Lakebase. The **Log action** button says *"not implemented yet"* — that's on purpose, you'll fix it next. 😉

> 🖱️ **What to look at in the UI** — *Compute ▸ Apps ▸ your app*:
> - **URL + status** — should read app `Running`, compute `Active`
> - **Deployments** tab — every deploy is versioned; open one to see its **build logs** (pip install, app start)
> - **Logs** tab — or append **`/logz`** to the app URL — the app's **runtime** output (your `print`s and errors land here; your first debugging stop)
> - **Resources** — the Lakebase DB you bound · **Permissions** — grant teammates `CAN_USE` to open it · **Environment** — the env vars from `app.yaml`

In [ ]:
# Deploy WITHOUT blocking the kernel (no *_and_wait), then poll politely so the notebook
# stays responsive and shows progress.
_prev = (w.apps.get(name=APP).as_dict().get("active_deployment") or {}).get("deployment_id")
w.apps.deploy(app_name=APP,
              app_deployment=AppDeployment.from_dict({"source_code_path": APP_SRC}))
print("⏳ Deploying your app — usually under a minute…")
_url = None
for _ in range(90):
    d = w.apps.get(name=APP).as_dict()
    ad = d.get("active_deployment") or {}
    state = (ad.get("status") or {}).get("state")
    if ad.get("deployment_id") != _prev and state == "SUCCEEDED":
        _url = d.get("url"); break
    if state == "FAILED":
        raise RuntimeError("Deploy failed: " + str((ad.get("status") or {}).get("message")))
    print("   …deploying")
    time.sleep(8)
print("✅ Deployed!")
print("🌐 Open your app:", _url or w.apps.get(name=APP).url)

## Step 5 — Finish the write-back (the payoff) ✍️

The app can *show* alerts but can't yet *save* a technician's work — because one function, `log_maintenance_action()` in `db.py`, is deliberately left blank (it raises `NotImplementedError`, which is why the button warned you).

Below is the finished version — read it (it's a single `INSERT`), then run the cell to drop it in. Want a challenge? Write it yourself first, then compare.

> 🧠 **What it does, in words:** add one row to `maintenance_actions` — which machine, which ticket, what was done, by whom, and (if the job is finished) the completion time.

> 💡 **Why a separate table?** The synced `maintenance_tickets` is a **read-only** replica — the app can't edit it. So the app records work in its **own** `maintenance_actions` table (in `public`), and CDF streams that back to the Lakehouse in Lab 4. That's the round-trip.

In [ ]:
# This is the ONE bit of logic that's yours to finish. Open bundle/src/app/db.py and look at
# log_maintenance_action() — right now it just raises NotImplementedError. Below is the finished
# version. (Try writing it yourself first if you like, then run this cell to drop it in.)
#
# In plain English: it inserts one row into the maintenance_actions table — which machine, which
# ticket, what was done, by whom, and (if the job is finished) the completion time.

impl = '''    with conn.cursor() as cur:
        cur.execute(
            """INSERT INTO maintenance_actions
                   (machine_id, ticket_id, action_type, description, performed_by, status, completed_at)
               VALUES (%s, %s, %s, %s, %s, %s, CASE WHEN %s = 'completed' THEN now() END)""",
            (machine_id, ticket_id, action_type, description, performed_by, status, status))
    conn.commit()'''

stub = '    raise NotImplementedError("Not implemented yet — see Lab 3, Step 4.")'
dbpy = pathlib.Path(f"{APP_SRC}/db.py")
src = dbpy.read_text()
if stub in src:
    dbpy.write_text(src.replace(stub, impl))
    print("✅ Write-back implemented — log_maintenance_action() now saves the technician's work.")
elif "INSERT INTO maintenance_actions" in src:
    print("✅ Write-back is already implemented — nothing to change. (Re-running is fine.)")
else:
    print("⚠️ Couldn't find the stub or an existing implementation in db.py — take a look before deploying.")

### …then redeploy 🎬

Push your change (same non-blocking deploy) and reload the app.

**✅ Check:** in the app, pick an alert, choose an action type, describe the fix, and hit **Log action**. It saves now — your entry shows under **"Recent maintenance actions."** You just closed the loop between the shop floor and the database. 🙌

In [ ]:
# Same non-blocking deploy as before — push your write-back and wait for it to go live.
_prev = (w.apps.get(name=APP).as_dict().get("active_deployment") or {}).get("deployment_id")
w.apps.deploy(app_name=APP,
              app_deployment=AppDeployment.from_dict({"source_code_path": APP_SRC}))
print("⏳ Redeploying with your change — usually under a minute…")
_url = None
for _ in range(90):
    d = w.apps.get(name=APP).as_dict()
    ad = d.get("active_deployment") or {}
    state = (ad.get("status") or {}).get("state")
    if ad.get("deployment_id") != _prev and state == "SUCCEEDED":
        _url = d.get("url"); break
    if state == "FAILED":
        raise RuntimeError("Deploy failed: " + str((ad.get("status") or {}).get("message")))
    print("   …deploying")
    time.sleep(8)
print("✅ Redeployed! Reload the app and try 'Log action' again.")
print("🌐", _url or w.apps.get(name=APP).url)

## Step 6 — Watch it run: logs & monitoring 👀

It's live — now know where to look, because this is the everyday operational surface for *any* App:

- **Runtime logs:** the app's **Logs** tab, or **`<app-url>/logz`** in the browser — your output and errors show here.
  > 💡 For durable, queryable logs in production, write to a **UC volume/table** or ship them to an APM (Datadog, etc.).
- **Deployments:** every deploy is versioned — open one to inspect its build logs, and **roll back any time** by redeploying an earlier version (check out the previous commit, or point at the older folder, and Deploy again).
- **Status & health:** the app's **compute status** on its details page.
- **Audit & cost (Databricks SQL):** `system.access.audit` (who created/deployed/started the app) and `system.billing.usage` (the app's DBUs).

> 💡 The loop you'll use forever: **edit → deploy → check Logs → check status.**

## Step 7 — Extend the app with the Databricks Assistant 🤖

You just did the **edit → deploy** loop by hand. Now do it with an **AI pair-programmer**. The **Databricks Assistant** is built into the workspace editor: describe a change in plain English, it writes the code, you review the diff, and deploy. You'll make **two** small changes to `app.py` — **without writing any code yourself**.

**Open the Assistant:** open **`bundle/src/app/app.py`** in the workspace file editor and click the **✨ Assistant** icon (or select a line and use the inline-edit shortcut). Type a prompt, review the suggested diff, and **Accept**.

**Prompt 1 — brand the app (warm-up):**
> Add the Databricks logo to the app using `st.logo` with this image URL: `https://cdn.jsdelivr.net/gh/homarr-labs/dashboard-icons/webp/databricks.webp`

You should get a single `st.logo(...)` line near the top; the logo then shows **top-left and in the sidebar**.

**Prompt 2 — visualize the queue (a bit meatier):**
> In the Alerts tab, add a small bar chart of the open alerts grouped by priority, just below the Open / High / Medium counters.

The Alerts tab already computes a `counts` dict (`high` / `medium` / `low`), so the Assistant has everything it needs — expect a couple of lines using `st.bar_chart`.

Then **run the redeploy cell below**, reload the app, and you'll see the **logo** and the **alerts chart**.

> 💡 **Prompting tip:** the Assistant reads your open file as context, so the more specific you are (function names, where to put it) the better the diff. Nothing changes until you **Accept** — if a suggestion looks off, tweak the prompt and retry. You're always in control.

> 🔎 **Stuck, or want to compare?** Good output looks roughly like this (pandas already ships with Streamlit, so there's no new dependency to add):
> ```python
> # near the top of app.py
> st.logo("https://cdn.jsdelivr.net/gh/homarr-labs/dashboard-icons/webp/databricks.webp")
>
> # in the Alerts tab, just after the counters
> import pandas as pd
> st.bar_chart(pd.Series(counts, name="open alerts"))
> ```

In [ ]:
# Redeploy your Assistant-made changes — same non-blocking deploy as Steps 4 & 5.
_prev = (w.apps.get(name=APP).as_dict().get("active_deployment") or {}).get("deployment_id")
w.apps.deploy(app_name=APP,
              app_deployment=AppDeployment.from_dict({"source_code_path": APP_SRC}))
print("⏳ Redeploying with your Assistant-made changes — usually under a minute…")
_url = None
for _ in range(90):
    d = w.apps.get(name=APP).as_dict()
    ad = d.get("active_deployment") or {}
    state = (ad.get("status") or {}).get("state")
    if ad.get("deployment_id") != _prev and state == "SUCCEEDED":
        _url = d.get("url"); break
    if state == "FAILED":
        raise RuntimeError("Deploy failed: " + str((ad.get("status") or {}).get("message")))
    print("   …deploying")
    time.sleep(8)
print("✅ Redeployed! Reload the app — you should see the Databricks logo and the alerts chart.")
print("🌐", _url or w.apps.get(name=APP).url)

## 🎉 You built, deployed, *and understood* a Databricks App

**What you now know:**
- An App is a **managed web app** on Databricks, behind SSO, running as its **own service principal**
- **Create** (provision + identity + resources) vs **Deploy** (ship code) — and you did both
- **Auth:** no passwords — short-lived **OAuth tokens per connection**; Lakebase reached via a **bound resource + grants**
- The app speaks **Postgres (Lakebase)**; the **Lakehouse** is the governed source (synced in Lab 2, fed back by CDF in Lab 4)
- Where to operate it in the **UI:** URL, Deployments, **Logs / `/logz`**, Resources, Permissions
- How to **extend it with the Databricks Assistant** — describe a change in plain English, review the diff, redeploy

**Why it's powerful:** the person on the floor gets a dead-simple app on your *governed* data, and their work flows straight back to analytics — no second system, no ETL, no secrets to manage.

➡️ **Next: [Lab 4 – Close the Round-Trip](../labs/Lab%204%20-%20Close%20the%20Round-Trip.md)** — watch your app's writes appear in Databricks SQL.

> 🧹 Want to run this again from scratch (or clean up)? Use the **`Lab3 - Reset (cleanup)`** notebook.